In [1]:
## ====================================
## EET-4501 – Applied Machine Learning
## Assignment 7
## ====================================

In [ ]:
## Part 1: Exploring the Data

!pip install tensorflow

import tensorflow as tf
from tensorflow.keras.datasets import mnist

(x_train, y_train), (x_test, y_test) = mnist.load_data()

print("Training images:", x_train.shape[0])
print("Testing images:", x_test.shape[0])

import numpy as np
num_classes = len(np.unique(y_train))
print("Number of classes:", num_classes)

import matplotlib.pyplot as plt

def show_image(image, label):
    plt.imshow(image, cmap='gray')
    plt.title(f"Label: {label}")
    plt.axis('off')
    plt.show()

## Display one image
show_image(x_train[0], y_train[0])

print("Image shape:", x_train[0].shape)

## Normalize
x_train_norm = x_train / 255.0
x_test_norm = x_test / 255.0

## Display the same image after normalization
show_image(x_train_norm[0], y_train[0])

In [ ]:
## Part 2: Build the Neural Network Model

from tensorflow.keras import models, layers

model = models.Sequential([
    layers.Flatten(input_shape=(28, 28)),
    layers.Dense(128, activation='tanh'),
    layers.Dense(128, activation='tanh'),
    layers.Dense(128, activation='tanh'),
    layers.Dense(10)  # Output layer
])

model.summary()

from tensorflow.keras.utils import plot_model

plot_model(model, to_file='Assign7_model.png', show_shapes=True, show_layer_names=True)

In [ ]:
## Part 3: Model Configuration

model_adam = tf.keras.models.clone_model(model)

model_adam.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

model_rmsprop = tf.keras.models.clone_model(model)

model_rmsprop.compile(
    optimizer='rmsprop',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

In [ ]:
## Part 4: Model Training

import datetime

log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

history_adam = model_adam.fit(
    x_train_norm, y_train,
    epochs=10,
    validation_data=(x_test_norm, y_test),
    callbacks=[tensorboard_callback]
)

history_rmsprop = model_rmsprop.fit(
    x_train_norm, y_train,
    epochs=10,
    validation_data=(x_test_norm, y_test),
    callbacks=[tensorboard_callback]
)

## Adam usually achieves better accuracy as:
## It adapts learning rates per parameter
## Has faster convergenceore
## Better stable training compared to RMSprop

In [ ]:
## Part 5: Evaluation and Analysis

test_loss_adam, test_acc_adam = model_adam.evaluate(x_test_norm, y_test)
test_loss_rms, test_acc_rms = model_rmsprop.evaluate(x_test_norm, y_test)

print("Adam Test Accuracy:", test_acc_adam)
print("RMSprop Test Accuracy:", test_acc_rms)

%load_ext tensorboard
%tensorboard --logdir logs/fit

In [ ]:
## Part 6: Model Testing

import numpy as np

## Softmax for probabilities
prob_model = tf.keras.Sequential([
    model_adam,
    tf.keras.layers.Softmax()
])

## Select 10 test images
indices = np.random.choice(len(x_test), 10)

for i in indices:
    img = x_test_norm[i]
    true_label = y_test[i]

    prediction = prob_model.predict(img.reshape(1,28,28))
    predicted_label = np.argmax(prediction)
    confidence = np.max(prediction)

    plt.imshow(img, cmap='gray')
    plt.title(f"Pred: {predicted_label}, True: {true_label}, Conf: {confidence:.2f}")
    plt.axis('off')
    plt.show()

In [ ]:
## Bonus

from PIL import Image
import numpy as np

# Load image
img = Image.open('bonus.png').convert('L')

# Resize
img = img.resize((28, 28))

# Convert to array
img_array = np.array(img)

# Invert if needed
img_array = 255 - img_array

# Normalize
img_array = img_array / 255.0

# Reshape
img_array = img_array.reshape(1, 28, 28)

# Predict
prediction = prob_model.predict(img_array)
predicted_label = np.argmax(prediction)

# Display
plt.imshow(img_array.reshape(28,28), cmap='gray')
plt.title(f"Predicted: {predicted_label}")
plt.axis('off')
plt.show()